In [0]:
# %sql
# truncate table telco_bronze.device_bronze_table;
# truncate table telco_bronze.device_silver_table;
# truncate table telco_gold.dim_device_gold_scd1

In [0]:
# %sql
# create schema if not exists telco_bronze;
# create schema if not exists telco_silver;
# create schema if not exists telco_gold;

In [0]:
device_schema_location = "abfss://source@sourcesystemadlsgen2.dfs.core.windows.net/_schemas/device_schema/"
device_bronze_checkpoint_location = "abfss://source@sourcesystemadlsgen2.dfs.core.windows.net/_checkpoints/device_bronze/"
device_silver_checkpoint_location = "abfss://source@sourcesystemadlsgen2.dfs.core.windows.net/_checkpoints/device_silver/"
bronze_table = "telco_bronze.device_bronze_table"
silver_table = "telco_bronze.device_silver_table"

configs ={
    "cloudFiles.format":"parquet",
    "header":"true",
    "cloudFiles.inferColumnTypes":"true",
    "cloudFiles.schemaLocation":f"{device_schema_location}",
    "cloudFiles.schemaEvolutionMode":"addNewColumns",
    "cloudFiles.maxFilesPerTrigger":"10"
}

In [0]:
from pyspark.sql.functions import * 

landing_df = spark.readStream.format("cloudFiles")\
    .options(**configs)\
        .load("abfss://source@sourcesystemadlsgen2.dfs.core.windows.net/device_source/*")

bronze_df = landing_df.withColumn("ingetsion_time", current_timestamp())\
    .withColumn("soure", lit("retail"))

bronze_df.writeStream.queryName("device_bronze_ingestion")\
    .format("delta")\
        .option("checkpointLocation",f"{device_bronze_checkpoint_location}")\
            .option("mergeSchema","true")\
                .outputMode("append")\
                    .toTable(f"{bronze_table}")


In [0]:
%sql
select * from telecom_7405608425992653.telco_bronze.device_bronze_table

In [0]:
bronze_stream_df = spark.readStream.table("telecom_7405608425992653.telco_bronze.device_bronze_table")

silver_df = bronze_stream_df

silver_df.writeStream.queryName("device_silver")\
    .format("delta")\
        .outputMode("append")\
            .option("checkpointLocation",f"{device_silver_checkpoint_location}")\
                .option("mergeSchema","true")\
                    .toTable(f"{silver_table}") 

In [0]:
%sql
select * from telecom_7405608425992653.telco_bronze.device_silver_table

In [0]:
spark.read.table(f"{silver_table}").where("""updated_at > coalesce((select max(updated_at) from telecom_7405608425992653.telco_gold.dim_device_gold_scd1),'1900-01-01')""").createOrReplaceTempView("silver_device_view")

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS telco_gold.dim_device_gold_scd1 (
#     device_id STRING, brand STRING, model STRING, os STRING, 
#     device_type STRING, owner_customer_id STRING, status STRING, updated_at TIMESTAMP
# ) USING DELTA TBLPROPERTIES (delta.enableChangeDataFeed = true);

# -- Gold SCD2 Table (Includes DLT-style history columns)
# CREATE TABLE IF NOT EXISTS telco_gold.dim_device_gold_scd2 (
#     device_id STRING, brand STRING, model STRING, os STRING, 
#     device_type STRING, owner_customer_id STRING, status STRING, updated_at TIMESTAMP,
#     start_at TIMESTAMP, end_at TIMESTAMP
# ) USING DELTA;

In [0]:
%sql
merge into telco_gold.dim_device_gold_scd1 t
using silver_device_view s
on s.device_id = t.device_id
when matched and s.status = "Inactive" then
delete
when matched and s.updated_at > t.updated_at and
    hash(s.device_type,s.brand,s.model,s.os,s.owner_customer_id,s.status) <>
    hash(t.device_type,t.brand,t.model,t.os,t.owner_customer_id,t.status) then
    update set *
when not matched and s.status != "Inactive" then
insert *

In [0]:
%sql
select * from telecom_7405608425992653.telco_gold.dim_device_gold_scd1